# 10 · 병렬 알고리즘 cuda-cccl (`cuda.compute`)

> **CuPy 2일 집중 코스 — Day 2 / 단원 7 (병렬 알고리즘, GTC 05 기반)**

08·09에서 커널을 **직접** 짰다면, 여기서는 NVIDIA가 제공하는 **검증된 병렬 알고리즘**으로
같은 일을 더 쉽고 빠르게 합니다. `cuda.compute`(cuda-cccl)의 reduce·scan·transform·sort 등은
손으로 튜닝한 CUDA 커널 수준의 성능을 Python에서 제공합니다.

## 이 노트북의 위치 (07 개념과의 관계)
- 07~09의 개념(인덱싱·coalescing·공유메모리·atomic)을 **알고리즘이 내부에서 처리** → 우리는 *무엇을* 할지만 지정
- 직접 커널(생산성↓·성능↑·제어↑) vs cccl(생산성↑·검증된 성능) 의 트레이드오프

## 학습 목표
- `reduce_into`로 합/커스텀 리덕션을 수행한다.
- 사용자 정의 이항연산·`unary_transform`·iterator를 활용한다.
- 언제 cccl / Numba / CuPy를 쓸지 판단한다.

## 목차
1. [cuda-cccl 이란 & 설치](#1)
2. [reduce_into — 합](#2)
3. [커스텀 이항연산 리덕션](#3)
4. [unary_transform](#4)
5. [Iterators (메모리 없는 시퀀스)](#5)
6. [언제 무엇을 쓰나](#6)
7. [스캔·커스텀·평가](#8) · 8. [연습](#7-ex)

> `cuda-cccl`은 **실험적 패키지**(2026 기준 `cuda.compute`, v0.5.x)입니다. API가 버전에 따라 바뀔 수 있습니다.

<a id="1"></a>
## 1. cuda-cccl 이란 & 설치

**CCCL**(CUDA Core Compute Libraries)의 Python 인터페이스 `cuda.compute` 는 배열/범위 단위 **병렬 알고리즘**을 제공합니다.
- reduce, scan, sort, transform, … (아키텍처 이식성 + 고성능)
- 사용자 정의 연산은 내부적으로 **numba-cuda**로 컴파일됩니다(람다 불가, 클로저 권장).

설치(강의장 환경): `pip install cuda-cccl`. 미설치 시 아래 셀은 안내만 출력합니다.

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, allclose
print_env()
try:
    import cuda.compute
    from cuda.compute import OpKind, CountingIterator, TransformIterator
    HAS_CCCL = True
    print('cuda.compute 사용 가능')
except Exception as e:
    HAS_CCCL = False
    print('cuda-cccl 미설치 — `pip install cuda-cccl` 후 실행하세요. (',e,')')

<a id="2"></a>
## 2. reduce_into — 합

`reduce_into(d_in, d_out, op, num_items, h_init)`. `op`은 내장 `OpKind.PLUS` 등.
`h_init`은 **host(numpy) 초기값 배열**, `d_out`은 결과를 담을 device 배열.

In [ ]:
dtype = np.int32
d_in  = cp.arange(1, 1_000_001, dtype=dtype)   # 1..1,000,000
d_out = cp.empty(1, dtype=dtype)
h_init = np.array([0], dtype=dtype)
cuda.compute.reduce_into(d_in, d_out, OpKind.PLUS, d_in.size, h_init)
print('cccl 합:', int(d_out[0]), '/ cupy 합:', int(d_in.sum()))
allclose(int(d_in.sum()), int(d_out[0]), name='reduce sum')

<a id="3"></a>
## 3. 커스텀 이항연산 리덕션

내장 연산 대신 **함수**를 넘기면 임의의 리덕션이 됩니다(람다 불가). 예: 짝수만 합산.

In [ ]:
def add_even(a, b):
    return (a if a % 2 == 0 else 0) + (b if b % 2 == 0 else 0)

d_in = cp.array([1,2,3,4,5,6], dtype=np.int32); d_out = cp.empty(1, dtype=np.int32)
cuda.compute.reduce_into(d_in, d_out, add_even, d_in.size, np.array([0],dtype=np.int32))
print('짝수 합:', int(d_out[0]))   # 2+4+6 = 12

<a id="4"></a>
## 4. unary_transform

각 원소에 함수를 적용해 출력 배열에 씁니다: `unary_transform(d_in, d_out, func, num_items)`.

In [ ]:
def sq(x): return x * x
d_in = cp.arange(10, dtype=np.int32); d_out = cp.empty(10, dtype=np.int32)
cuda.compute.unary_transform(d_in, d_out, sq, d_in.size)
print(cp.asnumpy(d_out))   # 0,1,4,9,...,81

<a id="5"></a>
## 5. Iterators — 메모리 없는 시퀀스

`CountingIterator`/`TransformIterator`는 **메모리를 할당하지 않고** 시퀀스를 표현해 알고리즘에 바로 넣습니다.
예: 10부터의 정수 100개의 합(배열 생성 없이).

In [ ]:
first = CountingIterator(np.int32(10))
d_out = cp.empty(1, dtype=np.int32)
cuda.compute.reduce_into(first, d_out, OpKind.PLUS, 100, np.array([0],dtype=np.int32))
print('합(10..109):', int(d_out[0]))

<a id="6"></a>
## 6. 언제 무엇을 쓰나

| 선택 | 쓸 때 |
|------|-------|
| **CuPy 함수**(`cp.sum`…) | 표준 연산, 가장 간단 |
| **`@cupy.fuse`/Elementwise/Reduction** | 원소·리덕션 융합, 약간의 커스텀 |
| **cuda-cccl**(`cuda.compute`) | reduce/scan/sort/transform + **커스텀 연산**을 검증된 성능으로 |
| **Numba CUDA** | 공유메모리·atomic·복잡한 인덱싱 등 **완전한 제어** |

정리: 표준은 CuPy, 정형 병렬패턴은 cccl, 특수 커널은 Numba.

<a id="8"></a>
## 7. 스캔 · 커스텀 타입 · 종합 평가

리덕션 외에 **스캔(prefix)**, **커스텀 구조체 타입**, 그리고 GTC 평가 과제를 다룹니다.
> 스캔 API는 실험적이라 버전에 따라 인자 순서가 다를 수 있습니다 — 공식 예제로 확인하세요.

### 8.1 스캔(prefix sum)

**exclusive scan**: `out[i] = op(in[0..i-1])` (자기 앞까지 누적). 누적합·러닝 통계의 기본.

In [ ]:
# 러닝 합 (exclusive scan, PLUS)
N = 10
d_in = cp.ones(N, dtype=np.int32); d_out = cp.empty(N, dtype=np.int32)
h_init = np.array([0], dtype=np.int32)
cuda.compute.exclusive_scan(d_in, d_out, OpKind.PLUS, h_init, N)
print(cp.asnumpy(d_out))   # 0,1,2,...,9 (앞까지의 합)

### 8.2 커스텀 구조체 타입 (`gpu_struct`)

사용자 정의 타입으로도 리덕션이 됩니다. 예: 픽셀들 중 **녹색(g)이 최대**인 픽셀 찾기(공식 예제).

In [ ]:
from cuda.compute import gpu_struct
@gpu_struct
class Pixel:
    r: np.int32
    g: np.int32
    b: np.int32
def max_g(x, y): return x if x.g > y.g else y
d_rgb = cp.random.randint(0,256,(10,3),dtype=np.int32).view(Pixel.dtype)
d_out = cp.empty(1, Pixel.dtype)
cuda.compute.reduce_into(d_rgb, d_out, max_g, d_rgb.size, Pixel(0,0,0))
print('max-green pixel:', d_out.get())

### 8.3 종합 평가 — 센서 '직전 최고치' (GTC 08 차용)

센서 온도 스트림에서 **각 시점 직전까지의 최고 온도**를 구하세요 — `max` 연산 **exclusive scan**으로.
`prev_peak[i] = max(readings[0..i-1])` (i=0은 초기값).

In [ ]:
def max_op(a, b): return a if a > b else b
N = 1_000_000
readings = cp.random.randint(0, 100, size=N, dtype=np.int32)
prev_peak = cp.empty(N, dtype=np.int32)
# TODO: h_init = np.array([-2**31], dtype=np.int32)   # 아주 작은 초기값
# TODO: cuda.compute.exclusive_scan(readings, prev_peak, max_op, h_init, N)
# 검증(작은 N): np.maximum.accumulate 로 비교

<details><summary>💡 해답 보기</summary>

```python
h_init = np.array([-2**31], dtype=np.int32)
cuda.compute.exclusive_scan(readings, prev_peak, max_op, h_init, N)
# 검증: prev_peak[i] == max(readings[:i])  (i>=1)
r = cp.asnumpy(readings)
ref = np.empty(N, dtype=np.int64); ref[0] = -2**31
ref[1:] = np.maximum.accumulate(r)[:-1]
assert (cp.asnumpy(prev_peak)[1:] == ref[1:]).all(); print('previous-peak OK')
```
</details>

<a id="7"></a>
## 8. 연습 — 커스텀 op 최댓값

**연습 — 커스텀 op로 최댓값**: `reduce_into`에 `max_op(a,b)`와 적절한 `h_init`(아주 작은 값)을 줘서 최댓값을 구하세요.

In [ ]:
def max_op(a, b):
    return a if a > b else b

d_in = cp.random.randint(0, 10**6, size=1_000_000, dtype=np.int32)
d_out = cp.empty(1, dtype=np.int32)
# TODO: h_init = np.array([np.iinfo(np.int32).min], dtype=np.int32)
# TODO: cuda.compute.reduce_into(d_in, d_out, max_op, d_in.size, h_init)
# allclose(int(d_in.max()), int(d_out[0]), name='reduce max')

<details><summary>💡 해답 보기</summary>

```python
h_init = np.array([np.iinfo(np.int32).min], dtype=np.int32)
cuda.compute.reduce_into(d_in, d_out, max_op, d_in.size, h_init)
allclose(int(d_in.max()), int(d_out[0]), name='reduce max')
```
</details>

### 체크포인트
- [ ] `reduce_into`로 합·커스텀 리덕션을 수행했다
- [ ] `unary_transform`·iterator를 사용했다
- [ ] cccl / Numba / CuPy의 사용 시점을 구분한다
- [ ] (참고) scan/sort 등 다른 알고리즘도 같은 방식으로 쓸 수 있음을 안다

다음: **`11_rawkernel`** — RawKernel(CUDA C)과 커널 개념 적용(최적화) 기술.